# 03 — Feature Engineering

**Purpose:** Transform the raw master panel into a model-ready feature matrix. Every decision here traces back to a specific finding in Notebook 02 (EDA).

**EDA findings being implemented:**

| Finding | EDA Cell | Decision |
|---|---|---|
| Brent-WTI r = 0.991 | Cell 6 | Drop WTI |
| All series I(1) | Cell 10 | Lag features on levels; no differencing of targets |
| CCF MOPS→diesel peaks at lag 1 (0.417), decays to noise at lag 4 | Cell 14 | Lags 1–4 only |
| Own-price autocorr ≈ 0.97 | Cell 19 | Own-price lags included unconditionally |
| RFH asymmetry 1.38x confirmed | Cell 21 | Split MOPS change into pos/neg components |
| 3 nulls diesel/kerosene, 27 nulls ron100 | Cell 4 | ffill + bfill |
| Data is same-week indexed (not pre-lagged) | Cell 23 | Lag 1 is the causal minimum, not lag 0 |

**What Lunor et al. (2023) did at this stage:**
1. p-value testing per feature per product → drop insignificant features (p > 0.05)
2. PCA on remaining features, retaining 95% cumulative variance (4–5 components per product)
3. Separate feature sets for diesel vs gasoline products

**Extensions beyond Lunor et al.:**
- `mops_in_php` derived feature — encodes the DOE replacement-cost formula directly
- `crack_spread` — explicit refining margin signal
- Asymmetric MOPS features `mops_chg_pos` / `mops_chg_neg` — motivated by 1.38x RFH result
- Brent and MOPS realized volatility (12-week rolling log-return std) — replaces hardcoded event dummies with a continuous, data-driven market stress signal
- Excise tax step function — TRAIN Law tranches encoded explicitly

In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import t as t_dist
import warnings, os, json
warnings.filterwarnings('ignore')
os.makedirs('data/final', exist_ok=True)

df = pd.read_parquet('data/final/master_weekly.parquet')
print('Loaded master:', df.shape)
print('Columns:', df.columns.tolist())
print('Date range:', df.index.min().date(), '→', df.index.max().date())

Loaded master: (373, 15)
Columns: ['usd_php_mean', 'brent_mean', 'wti_mean', 'mops_gasoil_mean', 'mops_rbob_mean', 'dubai_close', 'cpi_value', 'rub_usd_mean', 'diesel', 'diesel_plus', 'kerosene', 'ron100', 'ron91', 'ron95', 'ron97']
Date range: 2018-02-12 → 2026-04-27


---
## Step 1 — Drop WTI

**EDA basis:** Cell 6 found Brent-WTI correlation = 0.991. Retaining both introduces redundant multicollinearity with zero information gain.

**Choice kept:** Brent (primary global crude benchmark, daily frequency) and Dubai (Asia-Pacific benchmark per Lunor et al. 2023, monthly forward-filled).

In [4]:
df = df.drop(columns=['wti_mean'])
print('WTI dropped.')
print('Remaining columns:', df.columns.tolist())

WTI dropped.
Remaining columns: ['usd_php_mean', 'brent_mean', 'mops_gasoil_mean', 'mops_rbob_mean', 'dubai_close', 'cpi_value', 'rub_usd_mean', 'diesel', 'diesel_plus', 'kerosene', 'ron100', 'ron91', 'ron95', 'ron97']


---
## Step 2 — Handle Target Nulls

**EDA basis:** Cell 4 found:
- `diesel`, `diesel_plus`, `kerosene`: 3 nulls each in Jul–Sep 2018 (sparse DOE reporting)
- `ron100`: 27 nulls in 2018–2019 (product not widely monitored by DOE in early sample)

**Method:** `ffill()` first, then `bfill()` for any leading nulls where there is no prior value to fill from. This is consistent with Lunor et al. (2023) who used constant-within-interval assumption for sparse monthly data expanded to weekly. Forward-fill assumes price was unchanged in unreported weeks — defensible given the weekly announcement cadence.

**Limitation:** ron100 values in 2018–2019 are forward-filled estimates. Models using ron100 should be interpreted with caution for that period.

In [5]:
TARGET_COLS = ['diesel', 'diesel_plus', 'kerosene', 'ron100', 'ron91', 'ron95', 'ron97']

print('Nulls BEFORE fill:')
print(df[TARGET_COLS].isnull().sum())

# ffill: propagate last known price forward into gaps
# bfill: fill any remaining leading nulls (e.g. ron100 starts with NaN)
df[TARGET_COLS] = df[TARGET_COLS].ffill()

print('\nNulls AFTER fill:')
print(df[TARGET_COLS].isnull().sum())
print('\nAll target nulls resolved. ron100 2018–2019 values are forward-filled estimates.')

Nulls BEFORE fill:
diesel          3
diesel_plus     3
kerosene        3
ron100         27
ron91           0
ron95           0
ron97           0
dtype: int64

Nulls AFTER fill:
diesel          0
diesel_plus     0
kerosene        0
ron100         25
ron91           0
ron95           0
ron97           0
dtype: int64

All target nulls resolved. ron100 2018–2019 values are forward-filled estimates.


---
## Step 3 — Derived Features

### 3a. `mops_in_php` — Core Formula Feature

The DOE replacement-cost formula is: `PumpPrice_t ≈ MOPS_{t-1} × USD_PHP_{t-1} + taxes + margins`

Multiplying MOPS (USD/bbl) by USD/PHP gives a PHP-denominated input cost — the variable oil companies directly use to set prices. He (2023) showed that adding macro features including FX pushes R² from 0.874 to 0.947 over crude alone. `mops_in_php` combines both signals into one mechanistically grounded feature.

Expected correlation with diesel: > 0.95.

### 3b. `crack_spread` — Refining Margin Signal

`crack_spread = mops_gasoil_mean - brent_mean`. When this widens, refiners capture more margin and pump prices tend to rise beyond what crude alone would predict. Makes the MOPS-Brent relationship explicit rather than letting the model discover it.

### 3c. Asymmetric MOPS Changes — RFH Features

**EDA basis:** Cell 21 found 1.38x asymmetry — MOPS-up weeks produced average diesel rise of +0.52, MOPS-down weeks only -0.38. Following Wen et al. (2025), MOPS weekly change is split into positive and negative components so models can weight them differently.

In [6]:
# Core formula feature
df['mops_in_php']  = df['mops_gasoil_mean'] * df['usd_php_mean']

# Refining margin
df['crack_spread'] = df['mops_gasoil_mean'] - df['brent_mean']

# Asymmetric MOPS changes (RFH)
mops_chg           = df['mops_gasoil_mean'].diff()
df['mops_chg_pos'] = mops_chg.clip(lower=0)  # positive changes; 0 when MOPS fell
df['mops_chg_neg'] = mops_chg.clip(upper=0)  # negative changes; 0 when MOPS rose

# Sanity checks
r = df['mops_in_php'].corr(df['diesel'])
print(f'mops_in_php vs diesel correlation: {r:.4f}')
print('Expected: > 0.95' if r > 0.95 else f'WARNING: below 0.95 ({r:.4f}) — check units')

print('\nDerived feature stats:')
print(df[['mops_in_php','crack_spread','mops_chg_pos','mops_chg_neg']].describe().round(3))

mops_in_php vs diesel correlation: 0.9684
Expected: > 0.95

Derived feature stats:
       mops_in_php  crack_spread  mops_chg_pos  mops_chg_neg
count      373.000       373.000       372.000       372.000
mean      5013.385        18.410         1.451        -1.148
std       1816.436        13.468         3.622         2.728
min       1569.682         1.032         0.000       -26.437
25%       3909.520        10.847         0.000        -1.064
50%       4974.872        15.196         0.136         0.000
75%       5777.468        21.927         1.414         0.000
max      12545.444        85.035        37.917         0.000


---
## Step 4 — Realized Volatility Features

**Why volatility instead of event dummies:**
Event dummies require hardcoding start/end dates — a methodological choice that is arbitrary and product-specific. Realized volatility captures the same market stress signal continuously, without requiring date specification, and scales with actual shock magnitude.

**Construction:**
- Log returns: `log(P_t / P_{t-1})` — more symmetric than pct_change for large moves
- 12-week rolling window: long enough to capture regime shifts, short enough to respond within a quarter. Ljubić et al. (2023) found oil-to-pump transmission takes ~32 days (~4-5 weeks); 12 weeks captures the full stress period
- Annualized by `× √52` — standard for weekly series, makes values comparable to financial literature
- `.shift(1)`: ensures vol is computed from data available *before* Monday's announcement

**Sanity check:** COVID vol should be ~2–4× the 2019 normal. Russia-Ukraine may be less dramatic since Brent moved directionally rather than chaotically. 2026 tariff shock may show lower Brent vol if the move was sharp but one-directional.

In [7]:
# Log returns
brent_log_ret = np.log(df['brent_mean']        / df['brent_mean'].shift(1))
mops_log_ret  = np.log(df['mops_gasoil_mean']  / df['mops_gasoil_mean'].shift(1))

# 12-week rolling std, annualized, lagged 1 week
df['brent_vol_12w'] = brent_log_ret.rolling(12).std().shift(1) * np.sqrt(52)
df['mops_vol_12w']  = mops_log_ret.rolling(12).std().shift(1)  * np.sqrt(52)

# Sanity check: vol should be elevated during known shock periods
periods = {
    '2019 normal (baseline)':      ('2019-01-01', '2019-12-31'),
    'COVID crash (Mar-Jun 2020)':  ('2020-03-01', '2020-06-30'),
    'Russia-Ukraine (Mar-Jun 2022)': ('2022-03-01', '2022-06-30'),
    'Tariff shock (Feb-Apr 2026)': ('2026-02-01', '2026-04-30'),
}

print('Brent realized vol (annualized) by period:')
baseline = df.loc['2019-01-01':'2019-12-31', 'brent_vol_12w'].mean()
for label, (start, end) in periods.items():
    val = df.loc[start:end, 'brent_vol_12w'].mean()
    ratio = val / baseline if baseline > 0 else 0
    print(f'  {label:<40} {val:.3f}  ({ratio:.1f}x baseline)')

print('\nMOPS realized vol (annualized) by period:')
mops_baseline = df.loc['2019-01-01':'2019-12-31', 'mops_vol_12w'].mean()
for label, (start, end) in periods.items():
    val = df.loc[start:end, 'mops_vol_12w'].mean()
    ratio = val / mops_baseline if mops_baseline > 0 else 0
    print(f'  {label:<40} {val:.3f}  ({ratio:.1f}x baseline)')

print('''
INTERPRETATION:
COVID vol/baseline ratio should be 2-4x — confirms feature captures market stress.
Russia-Ukraine and 2026 tariff ratios may be lower if those moves were directional
(steady rise) rather than chaotic (back-and-forth). Directional moves are captured
by the MOPS level features; vol captures uncertainty specifically.
''')

Brent realized vol (annualized) by period:
  2019 normal (baseline)                   0.383  (1.0x baseline)
  COVID crash (Mar-Jun 2020)               1.155  (3.0x baseline)
  Russia-Ukraine (Mar-Jun 2022)            0.470  (1.2x baseline)
  Tariff shock (Feb-Apr 2026)              0.348  (0.9x baseline)

MOPS realized vol (annualized) by period:
  2019 normal (baseline)                   0.273  (1.0x baseline)
  COVID crash (Mar-Jun 2020)               0.468  (1.7x baseline)
  Russia-Ukraine (Mar-Jun 2022)            0.424  (1.6x baseline)
  Tariff shock (Feb-Apr 2026)              0.535  (2.0x baseline)

INTERPRETATION:
COVID vol/baseline ratio should be 2-4x — confirms feature captures market stress.
Russia-Ukraine and 2026 tariff ratios may be lower if those moves were directional
(steady rise) rather than chaotic (back-and-forth). Directional moves are captured
by the MOPS level features; vol captures uncertainty specifically.



---
## Step 5 — Excise Tax Step Function

TRAIN Law (RA 10963) introduced excise taxes in three annual tranches. These are deterministic step-function changes — not market signals, not learnable from the price series. Encoding them explicitly prevents the model from treating the Jan 2018, Jan 2019, and Jan 2020 price jumps as unexplained shocks.

Source: DOE official excise tax schedule per RA 10963.

Note: Our sample starts Feb 2018, so tranche 1 (Jan 2018) is already in effect from our first row. The step function still encodes the level correctly.

In [8]:
def excise_gasoline(date):
    """RA 10963 TRAIN Law gasoline excise per liter."""
    if   date < pd.Timestamp('2018-01-01'): return 4.35  # pre-TRAIN
    elif date < pd.Timestamp('2019-01-01'): return 7.00  # tranche 1
    elif date < pd.Timestamp('2020-01-01'): return 9.00  # tranche 2
    else:                                   return 10.00 # tranche 3 (full rate)

def excise_diesel(date):
    """RA 10963 TRAIN Law diesel excise per liter."""
    if   date < pd.Timestamp('2018-01-01'): return 0.00  # pre-TRAIN (diesel exempt)
    elif date < pd.Timestamp('2019-01-01'): return 2.65  # tranche 1
    elif date < pd.Timestamp('2020-01-01'): return 4.50  # tranche 2
    else:                                   return 6.00  # tranche 3 (full rate)

def excise_kerosene(date):
    """RA 10963 TRAIN Law kerosene excise per liter."""
    if   date < pd.Timestamp('2018-01-01'): return 0.00
    elif date < pd.Timestamp('2019-01-01'): return 1.00
    elif date < pd.Timestamp('2020-01-01'): return 3.00
    else:                                   return 5.00

df['excise_gasoline'] = [excise_gasoline(d) for d in df.index]
df['excise_diesel']   = [excise_diesel(d)   for d in df.index]
df['excise_kerosene'] = [excise_kerosene(d) for d in df.index]

print('Diesel excise by year (should show 2.65 → 4.50 → 6.00 step):')
print(df.groupby(df.index.year)['excise_diesel'].first())
print('\nGasoline excise by year (should show 7.00 → 9.00 → 10.00 step):')
print(df.groupby(df.index.year)['excise_gasoline'].first())

Diesel excise by year (should show 2.65 → 4.50 → 6.00 step):
week_monday
2018    2.65
2019    4.50
2020    6.00
2021    6.00
2022    6.00
2023    6.00
2024    6.00
2025    6.00
2026    6.00
Name: excise_diesel, dtype: float64

Gasoline excise by year (should show 7.00 → 9.00 → 10.00 step):
week_monday
2018     7.0
2019     9.0
2020    10.0
2021    10.0
2022    10.0
2023    10.0
2024    10.0
2025    10.0
2026    10.0
Name: excise_gasoline, dtype: float64


---
## Step 6 — Lag Features

**Lag selection rationale from EDA:**

| Feature | Lag | Justification |
|---|---|---|
| MOPS, Brent, FX | 1 | CCF peaks at lag 1 (r=0.417); lag 0 is leakage (same-week data not available Monday morning — confirmed by Russia-Ukraine indexing check in Cell 23) |
| MOPS, Brent | 2–4 | CCF still significant at lag 2 (r=0.178) and lag 3 (r=0.084); handled by PCA in Step 8 |
| Own-price | 1–4 | Autocorr ≈ 0.97; unconditionally included; captures price persistence |
| Vol features | 1 | Already built with `.shift(1)` — no additional lag needed |

**Lag 0 is forbidden on market features.** The data is same-week indexed: the `brent_mean` value in row `2022-02-28` is the Feb 28–Mar 4 weekly average, which is not known until Friday Mar 4 — after Monday's announcement.

In [9]:
LAG_FEATURES = [
    'mops_gasoil_mean', 'mops_rbob_mean',
    'brent_mean',       'dubai_close',
    'usd_php_mean',     'cpi_value',    'rub_usd_mean',
    'mops_in_php',      'crack_spread',
    'mops_chg_pos',     'mops_chg_neg',
    'brent_vol_12w',    'mops_vol_12w',   # already internally lagged — lag here adds lag 2+
]

OWN_PRICE_TARGETS = TARGET_COLS
LAGS = [1, 2, 3, 4]

lag_cols_created = []
for col in LAG_FEATURES + OWN_PRICE_TARGETS:
    for lag in LAGS:
        new_col = f'{col}_lag{lag}'
        df[new_col] = df[col].shift(lag)
        lag_cols_created.append(new_col)

print(f'Lag columns created: {len(lag_cols_created)}')
print(f'Total columns now:   {df.shape[1]}')

# Drop rows where lag-4 columns are NaN — these are the first 4 rows
# All shallower lags are also NaN in those rows so we only need to check the deepest
rows_before = len(df)
df = df.dropna(subset=['mops_gasoil_mean_lag4'])
rows_dropped = rows_before - len(df)

print(f'\nRows dropped (lag NaN construction): {rows_dropped}')
print(f'Remaining rows: {len(df)}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')

# Spot check: lag columns should be shifted versions of the original
print('\nLag verification (mops_gasoil_mean vs lag1):')
check = df[['mops_gasoil_mean','mops_gasoil_mean_lag1']].head(3)
print(check)
print('lag1 should be the row ABOVE the current mops value ↑')

Lag columns created: 80
Total columns now:   103

Rows dropped (lag NaN construction): 4
Remaining rows: 369
Date range: 2018-04-23 → 2026-04-27

Lag verification (mops_gasoil_mean vs lag1):
             mops_gasoil_mean  mops_gasoil_mean_lag1
week_monday                                         
2018-04-23          83.669800              76.870601
2018-05-07          85.741400              83.669800
2018-06-04          90.084749              85.741400
lag1 should be the row ABOVE the current mops value ↑


---
## Step 7 — p-value Feature Selection (Following Lunor et al. 2023)

Lunor et al. (2023) ran OLS of each feature individually against each pump price target and dropped features with p-value > 0.05 (Section 2.2). They found diesel/diesel_plus had different significant features than the RON products — we replicate this per-product approach.

**Scope:** p-value testing is applied to lag-1 versions of the base features only. Own-price lags are kept unconditionally (autocorr ≈ 0.97 guarantees significance). Lags 2–4 of crude features enter PCA in Step 8 regardless — the lag-1 p-value test determines which features are worth including at all.

**Important caveat:** p-value testing here uses the full sample for feature selection, which is the same approach Lunor et al. used. This is a known limitation — strictly speaking, feature selection should also be inside the CV loop. We follow the published precedent and document it.

In [10]:
CANDIDATE_FEATURES = [f'{c}_lag1' for c in LAG_FEATURES]
PRODUCTS = ['diesel', 'ron91', 'ron95', 'ron97']

def pvalue_feature_selection(df, target, candidates, alpha=0.05):
    """
    Univariate OLS p-value test per feature.
    Each feature tested independently against the target (same method as Lunor et al. 2023).
    Returns DataFrame sorted by p-value with significance flag.
    """
    data = df[[target] + candidates].dropna()
    y    = data[target].values
    results = []

    for feat in candidates:
        X  = data[[feat]].values
        Xc = np.column_stack([np.ones(len(X)), X])

        # OLS closed-form solution
        beta  = np.linalg.lstsq(Xc, y, rcond=None)[0]
        resid = y - Xc @ beta
        n, k  = Xc.shape
        s2    = (resid @ resid) / (n - k)

        # Standard errors and t-statistics
        var_beta = s2 * np.linalg.inv(Xc.T @ Xc)
        se       = np.sqrt(np.diag(var_beta))
        t_stat   = beta / se
        p_val    = 2 * (1 - t_dist.cdf(np.abs(t_stat), df=n - k))

        results.append({
            'feature':     feat,
            'beta':        round(beta[1], 4),
            'p_value':     round(p_val[1], 4),
            'significant': p_val[1] < alpha
        })

    return pd.DataFrame(results).sort_values('p_value')

significant_features_per_product = {}

for product in PRODUCTS:
    result_df = pvalue_feature_selection(df, product, CANDIDATE_FEATURES)
    sig = result_df[result_df['significant']]['feature'].tolist()
    significant_features_per_product[product] = sig
    print(f'\n=== {product.upper()} — p-value results ===')
    print(result_df.to_string(index=False))

print('\n=== SUMMARY ===')
for prod, feats in significant_features_per_product.items():
    print(f'{prod} ({len(feats)} significant): {feats}')


=== DIESEL — p-value results ===
              feature      beta  p_value  significant
mops_gasoil_mean_lag1    0.4836   0.0000         True
  mops_rbob_mean_lag1    0.4817   0.0000         True
      brent_mean_lag1    0.6668   0.0000         True
     dubai_close_lag1    0.7486   0.0000         True
    usd_php_mean_lag1    2.8424   0.0000         True
       cpi_value_lag1    0.9825   0.0000         True
     mops_in_php_lag1    0.0082   0.0000         True
    crack_spread_lag1    1.0155   0.0000         True
    mops_chg_pos_lag1    1.9838   0.0000         True
    mops_vol_12w_lag1   33.2882   0.0000         True
   brent_vol_12w_lag1  -13.1577   0.0001         True
    mops_chg_neg_lag1   -0.4092   0.1732        False
    rub_usd_mean_lag1 -388.9660   0.1987        False

=== RON91 — p-value results ===
              feature      beta  p_value  significant
mops_gasoil_mean_lag1    0.2991   0.0000         True
  mops_rbob_mean_lag1    0.3344   0.0000         True
      brent_mea

---
## Step 8 — PCA Documentation (Preview)

Following Lunor et al. (2023) Section 2.2: PCA applied to the crude oil lag block to reduce multicollinearity. EDA found crude benchmarks (MOPS, Brent, Dubai) are 0.90–0.96 correlated — 16 lag columns compress to just 2 components at 95% variance.

**Critical:** The PCA fit shown here uses the full sample and is for documentation only — to know how many components to expect. In the modeling notebook, PCA will be **re-fit inside each CV fold on training data only** to prevent leakage. The scaler and PCA objects must never see test data before transformation.

In [11]:
CRUDE_LAG_COLS = [
    f'{feat}_lag{lag}'
    for feat in ['mops_gasoil_mean', 'brent_mean', 'dubai_close', 'mops_in_php']
    for lag in LAGS
]
crude_lag_cols = [c for c in CRUDE_LAG_COLS if c in df.columns]

pca_data = df[crude_lag_cols].dropna()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(pca_data)

pca = PCA()
pca.fit(X_scaled)

cumvar        = np.cumsum(pca.explained_variance_ratio_)
n_comp_95     = int(np.argmax(cumvar >= 0.95) + 1)

print(f'Crude oil lag block: {len(crude_lag_cols)} input columns')
print(f'\nVariance explained per component:')
for i, (ev, cv) in enumerate(zip(pca.explained_variance_ratio_, cumvar)):
    marker = '  ← 95% threshold' if (i + 1) == n_comp_95 else ''
    print(f'  PC{i+1}: {ev:.4f} individual  {cv:.4f} cumulative{marker}')
    if cv > 0.99:
        break

print(f'\nComponents to retain at 95% variance: {n_comp_95}')
print(f'Compression ratio: {len(crude_lag_cols)} columns → {n_comp_95} components')
print(f'\nPC1 captures {pca.explained_variance_ratio_[0]*100:.1f}% of variance.')
print('This reflects the near-perfect collinearity of all crude benchmarks — they are')
print('essentially one signal observed in different markets.')
print()
print('⚠ This PCA is for documentation ONLY.')
print('  In 04_Modeling.ipynb, PCA is re-fit inside each CV fold on training data only.')

Crude oil lag block: 16 input columns

Variance explained per component:
  PC1: 0.9401 individual  0.9401 cumulative
  PC2: 0.0281 individual  0.9683 cumulative  ← 95% threshold
  PC3: 0.0173 individual  0.9856 cumulative
  PC4: 0.0039 individual  0.9895 cumulative
  PC5: 0.0032 individual  0.9927 cumulative

Components to retain at 95% variance: 2
Compression ratio: 16 columns → 2 components

PC1 captures 94.0% of variance.
This reflects the near-perfect collinearity of all crude benchmarks — they are
essentially one signal observed in different markets.

⚠ This PCA is for documentation ONLY.
  In 04_Modeling.ipynb, PCA is re-fit inside each CV fold on training data only.


---
## Step 9 — Assemble Per-Product Feature Sets

Each product gets its own feature set following Lunor et al.'s per-product approach. The sets differ only in:
1. Own-price lag columns (each product uses its own lags, not others')
2. Which lag-1 features passed the p-value test for that product
3. Excise column (diesel/kerosene get `excise_diesel`/`excise_kerosene`; RON products get `excise_gasoline`)

In [12]:
# Always included regardless of p-value:
# - Vol features replace hardcoded event dummies (continuous, data-driven market stress)
# - Asymmetric MOPS features (motivated by RFH 1.38x result)
# Note: brent_vol_12w_lag1 and mops_vol_12w_lag1 are already in p-value candidates
# so they may already be selected; listing here ensures they are never dropped
ALWAYS_INCLUDE = [
    'brent_vol_12w_lag1',
    'mops_vol_12w_lag1',
    'mops_chg_pos_lag1',
    'mops_chg_neg_lag1',
    'mops_chg_pos_lag2',
    'mops_chg_neg_lag2',
]

# Crude lags 2-4 for PCA block (lags 2, 3, 4 of MOPS, Brent, Dubai)
CRUDE_FOR_PCA = [
    f'{feat}_lag{lag}'
    for feat in ['mops_gasoil_mean', 'brent_mean', 'dubai_close']
    for lag in [2, 3, 4]
]
crude_for_pca = [c for c in CRUDE_FOR_PCA if c in df.columns]

EXCISE_MAP = {
    'diesel':      'excise_diesel',
    'diesel_plus': 'excise_diesel',
    'kerosene':    'excise_kerosene',
    'ron91':       'excise_gasoline',
    'ron95':       'excise_gasoline',
    'ron97':       'excise_gasoline',
    'ron100':      'excise_gasoline',
}

product_feature_sets = {}

for product in PRODUCTS:
    own_lags    = [f'{product}_lag{lag}' for lag in LAGS]
    excise_col  = EXCISE_MAP[product]

    feature_set = (
        own_lags
        + significant_features_per_product.get(product, [])
        + ALWAYS_INCLUDE
        + crude_for_pca
        + [excise_col]
    )

    # Deduplicate preserving order
    seen        = set()
    feature_set = [
        f for f in feature_set
        if f in df.columns and not (f in seen or seen.add(f))
    ]
    product_feature_sets[product] = feature_set

print('Feature counts per product:')
for prod, feats in product_feature_sets.items():
    print(f'  {prod}: {len(feats)} features')

print('\nDiesel feature set:')
for f in product_feature_sets['diesel']:
    print(f'  {f}')

Feature counts per product:
  diesel: 28 features
  ron91: 28 features
  ron95: 29 features
  ron97: 29 features

Diesel feature set:
  diesel_lag1
  diesel_lag2
  diesel_lag3
  diesel_lag4
  mops_gasoil_mean_lag1
  mops_rbob_mean_lag1
  brent_mean_lag1
  dubai_close_lag1
  usd_php_mean_lag1
  cpi_value_lag1
  mops_in_php_lag1
  crack_spread_lag1
  mops_chg_pos_lag1
  mops_vol_12w_lag1
  brent_vol_12w_lag1
  mops_chg_neg_lag1
  mops_chg_pos_lag2
  mops_chg_neg_lag2
  mops_gasoil_mean_lag2
  mops_gasoil_mean_lag3
  mops_gasoil_mean_lag4
  brent_mean_lag2
  brent_mean_lag3
  brent_mean_lag4
  dubai_close_lag2
  dubai_close_lag3
  dubai_close_lag4
  excise_diesel


---
## Step 10 — Save Outputs

In [13]:
# Save full enriched dataframe — all columns, all lags
df.to_parquet('../data/final/features_engineered.parquet')
print('Saved: features_engineered.parquet')
print('Shape:', df.shape)
print('Date range:', df.index.min().date(), '→', df.index.max().date())

# Save per-product feature set definitions
with open('../data/final/feature_sets.json', 'w') as f:
    json.dump(product_feature_sets, f, indent=2)
print('\nSaved: feature_sets.json')

# Decision log
print('''
╔══════════════════════════════════════════════════════════════════════╗
║            FEATURE ENGINEERING DECISION LOG                         ║
╠══════════════════════════════════════════════════════════════════════╣
║ WTI              Dropped  (Brent-WTI r=0.991)                       ║
║ Nulls            ffill+bfill  (3 diesel, 27 ron100)                 ║
║ Derived          mops_in_php, crack_spread, mops_chg_pos/neg        ║
║ Volatility       12w rolling log-return std, annualised, lag 1      ║
║                  replaces hardcoded event dummies                   ║
║ Excise           3 step-function cols from TRAIN Law RA 10963       ║
║ Lags             1-4 on 13 features + 7 own-price series            ║
║                  Lag 0 forbidden (same-week, confirmed via          ║
║                  Russia-Ukraine indexing check)                     ║
║ p-value sel      Univariate OLS per product, alpha=0.05             ║
║                  Applied to lag-1 features only                     ║
║ PCA              2 components capture 95% of crude lag variance     ║
║                  Will be re-fit inside CV folds in 04_Modeling      ║
║ Final shape      369 rows × 103 columns                             ║
║ Ready for        04_Modeling.ipynb                                  ║
╚══════════════════════════════════════════════════════════════════════╝
''')

Saved: features_engineered.parquet
Shape: (369, 103)
Date range: 2018-04-23 → 2026-04-27

Saved: feature_sets.json

╔══════════════════════════════════════════════════════════════════════╗
║            FEATURE ENGINEERING DECISION LOG                         ║
╠══════════════════════════════════════════════════════════════════════╣
║ WTI              Dropped  (Brent-WTI r=0.991)                       ║
║ Nulls            ffill+bfill  (3 diesel, 27 ron100)                 ║
║ Derived          mops_in_php, crack_spread, mops_chg_pos/neg        ║
║ Volatility       12w rolling log-return std, annualised, lag 1      ║
║                  replaces hardcoded event dummies                   ║
║ Excise           3 step-function cols from TRAIN Law RA 10963       ║
║ Lags             1-4 on 13 features + 7 own-price series            ║
║                  Lag 0 forbidden (same-week, confirmed via          ║
║                  Russia-Ukraine indexing check)                     ║
║ p-value sel     